## Model Versioning

Every trained model gets its own folder with metadata, so we can always trace a prediction back to exactly which model made it.

```
models/
v1/
attrition_pipeline.joblib
metadata.json
v2/
attrition_pipeline.joblib
metadata.json
```

In [8]:
import os
import json
import joblib
import shutil
from datetime import date

## 1. Load Current Model and Metrics

Load the model and metrics from Notebook 07.

In [9]:
MODELS_PATH = '../models'

pipeline = joblib.load(f'{MODELS_PATH}/attrition_pipeline.joblib')
with open(f'{MODELS_PATH}/attrition_metrics.json') as f:
    metrics = json.load(f)

print('Current model metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

Current model metrics:
  Model: Random Forest
  Precision: 0.1935483870967742
  Recall: 0.5454545454545454
  F1: 0.2857142857142857
  ROC-AUC: 0.7875383043922369
  cv_roc_auc: 0.7934627170582227
  cv_roc_auc_std: 0.024873105217935697
  trained_on: train_set_only (400 samples)
  performance_rating_removed: True


## 2. Create Versioned Model Folder

Move the current model into a versioned folder structure.

In [10]:
version = 'v1'
version_dir = f'{MODELS_PATH}/{version}'
os.makedirs(version_dir, exist_ok=True)

# Copy pipeline to versioned folder
shutil.copy(f'{MODELS_PATH}/attrition_pipeline.joblib', f'{version_dir}/attrition_pipeline.joblib')

print(f'Created {version_dir}/')
print(f'Contents: {os.listdir(version_dir)}')

Created ../models/v1/
Contents: ['metadata.json', 'attrition_pipeline.joblib']


## 3. Create metadata.json

The metadata file follows the teacher's schema. This is what Notebook 09 (and any future notebook) will read to know which model to use.

In [11]:
metadata = {
    'model_name': 'Attrition Prediction Model',
    'version': version,
    'algorithm': 'Random Forest',
    'training_date': str(date.today()),
    'roc_auc': round(metrics.get('ROC-AUC', 0), 4),
    'f1_score': round(metrics.get('F1', 0), 4)
}

with open(f'{version_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('metadata.json:')
print(json.dumps(metadata, indent=2))

metadata.json:
{
  "model_name": "Attrition Prediction Model",
  "version": "v1",
  "algorithm": "Random Forest",
  "training_date": "2026-08-31",
  "roc_auc": 0.7875,
  "f1_score": 0.2857
}


## 4. Verify Versioned Model

Load the model back from the versioned folder to confirm it works.

In [12]:
# Load from versioned path
loaded_pipeline = joblib.load(f'{version_dir}/attrition_pipeline.joblib')
with open(f'{version_dir}/metadata.json') as f:
    loaded_metadata = json.load(f)

print(f'Loaded model: {loaded_metadata["algorithm"]} {loaded_metadata["version"]}')
print(f'Trained on: {loaded_metadata["training_date"]}')
print(f'ROC-AUC: {loaded_metadata["roc_auc"]}')
print(f'F1: {loaded_metadata["f1_score"]}')
print(f'Steps: {[name for name, _ in loaded_pipeline.steps]}')

Loaded model: Random Forest v1
Trained on: 2026-08-31
ROC-AUC: 0.7875
F1: 0.2857
Steps: ['columntransformer', 'randomforestclassifier']


## 5. Version Log

Maintain a log of all model versions for quick reference.

In [13]:
version_log_path = f'{MODELS_PATH}/version_log.json'

# Load existing log or create new one
if os.path.exists(version_log_path):
    with open(version_log_path) as f:
        version_log = json.load(f)
else:
    version_log = []

# Add this version
version_entry = {
    'version': metadata['version'],
    'algorithm': metadata['algorithm'],
    'training_date': metadata['training_date'],
    'roc_auc': metadata['roc_auc'],
    'f1_score': metadata['f1_score']
}

# Avoid duplicates
version_log = [v for v in version_log if v['version'] != version]
version_log.append(version_entry)

with open(version_log_path, 'w') as f:
    json.dump(version_log, f, indent=2)

print('Version log:')
print(json.dumps(version_log, indent=2))

Version log:
[
  {
    "version": "v1",
    "algorithm": "Random Forest",
    "training_date": "2026-08-31",
    "roc_auc": 0.7875,
    "f1_score": 0.2857
  }
]


## 6. Cleanup

Remove the old flat model files (now that they're versioned).

In [14]:
# Keep the flat files for now (Notebooks 06/07 expect them)
# Just confirm the versioned copy exists
print('Versioned model folder:')
print(f'  {version_dir}/')
for f in os.listdir(version_dir):
    size = os.path.getsize(f'{version_dir}/{f}')
    print(f'    {f} ({size:,} bytes)')

Versioned model folder:
  ../models/v1/
    metadata.json (175 bytes)
    attrition_pipeline.joblib (284,747 bytes)


## Summary

### What We Did
We versioned the current model (v1) by:

1. Creating a `models/v1/` folder
2. Copying the trained pipeline there
3. Writing `metadata.json` with the teacher's schema
4. Maintaining a `version_log.json` for quick reference

### Why Manual Versioning First
MLflow is powerful but adds complexity. Before reaching for it, we need:

- A clear versioning habit (done)
- A schema for model metadata (done)
- A way to trace predictions to models (done)

Once manual versioning feels limiting, MLflow is the natural next step — it tracks every training run's parameters, metrics, and artifacts automatically.

### Files Created
- `models/v1/attrition_pipeline.joblib` — trained model
- `models/v1/metadata.json` — model metadata (teacher's schema)
- `models/version_log.json` — all versions in one place